In [3]:
import sympy as sp

# =========================
# Symbols
# =========================

dt = sp.symbols('dt')
g = sp.symbols('g')

x, y, z = sp.symbols('x y z')
phi, theta, psi = sp.symbols('phi theta psi')
vx, vy, vz = sp.symbols('vx vy vz')

ax, ay, az = sp.symbols('ax ay az')
wx, wy, wz = sp.symbols('wx wy wz')


# =========================
# State
# =========================

state = sp.Matrix([
    x, y, z,
    phi, theta, psi,
    vx, vy, vz
])


# =========================
# IMU measurements
# =========================

a = sp.Matrix([
    ax,
    ay,
    az
])

omega = sp.Matrix([
    wx,
    wy,
    wz
])

gravity = sp.Matrix([
    0,
    0,
    -g
])


# =========================
# Rotation matrix R
# ZYX convention
# =========================

Rx = sp.Matrix([
    [1, 0, 0],
    [0, sp.cos(phi), -sp.sin(phi)],
    [0, sp.sin(phi),  sp.cos(phi)]
])

Ry = sp.Matrix([
    [ sp.cos(theta), 0, sp.sin(theta)],
    [0,              1, 0],
    [-sp.sin(theta), 0, sp.cos(theta)]
])

Rz = sp.Matrix([
    [sp.cos(psi), -sp.sin(psi), 0],
    [sp.sin(psi),  sp.cos(psi), 0],
    [0,             0,            1]
])

R = Rz * Ry * Rx


# =========================
# Euler angle rate matrix
# =========================

E = sp.Matrix([
    [
        1,
        sp.sin(phi) * sp.tan(theta),
        sp.cos(phi) * sp.tan(theta)
    ],
    [
        0,
        sp.cos(phi),
        -sp.sin(phi)
    ],
    [
        0,
        sp.sin(phi) / sp.cos(theta),
        sp.cos(phi) / sp.cos(theta)
    ]
])


# =========================
# Motion equations
# =========================

acc_world = R * a + gravity

position = sp.Matrix([x, y, z])
orientation = sp.Matrix([phi, theta, psi])
velocity = sp.Matrix([vx, vy, vz])


f_position = (
    position
    + velocity * dt
    + sp.Rational(1, 2) * acc_world * dt**2
)

f_orientation = (
    orientation
    + E * omega * dt
)

f_velocity = (
    velocity
    + acc_world * dt
)


# =========================
# Complete state transition
# =========================

f = sp.Matrix.vstack(
    f_position,
    f_orientation,
    f_velocity
)


# =========================
# Print f(x,u)
# =========================

# sp.print_latex(f)

print("\nf(x,u) =")
f


f(x,u) =


Matrix([
[ dt**2*(ax*cos(psi)*cos(theta)/2 + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))/2 + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2) + dt*vx + x],
[dt**2*(ax*sin(psi)*cos(theta)/2 + ay*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi))/2 + az*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))/2) + dt*vy + y],
[                                                                 dt**2*(-ax*sin(theta)/2 + ay*sin(phi)*cos(theta)/2 + az*cos(phi)*cos(theta)/2 - g/2) + dt*vz + z],
[                                                                                                  dt*(wx + wy*sin(phi)*tan(theta) + wz*cos(phi)*tan(theta)) + phi],
[                                                                                                                           dt*(wy*cos(phi) - wz*sin(phi)) + theta],
[                                                                                                       dt*(wy*sin(phi)/cos(theta) + wz*cos(phi)/cos(theta)) + psi],
[

In [4]:
F = f.jacobian(state)

print("\nF =")
F


F =


Matrix([
[1, 0, 0,  dt**2*(ay*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2 + az*(-sin(phi)*sin(theta)*cos(psi) + sin(psi)*cos(phi))/2), dt**2*(-ax*sin(theta)*cos(psi)/2 + ay*sin(phi)*cos(psi)*cos(theta)/2 + az*cos(phi)*cos(psi)*cos(theta)/2), dt**2*(-ax*sin(psi)*cos(theta)/2 + ay*(-sin(phi)*sin(psi)*sin(theta) - cos(phi)*cos(psi))/2 + az*(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))/2), dt,  0,  0],
[0, 1, 0, dt**2*(ay*(-sin(phi)*cos(psi) + sin(psi)*sin(theta)*cos(phi))/2 + az*(-sin(phi)*sin(psi)*sin(theta) - cos(phi)*cos(psi))/2), dt**2*(-ax*sin(psi)*sin(theta)/2 + ay*sin(phi)*sin(psi)*cos(theta)/2 + az*sin(psi)*cos(phi)*cos(theta)/2),   dt**2*(ax*cos(psi)*cos(theta)/2 + ay*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi))/2 + az*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))/2),  0, dt,  0],
[0, 0, 1,                                                                 dt**2*(ay*cos(phi)*cos(theta)/2 - az*sin(phi)*cos(theta)/2),                            dt**2*(-ax*cos(

In [7]:
"""
Symbolic derivation of the robot_localization EKF motion model
(constant-acceleration, constant-angular-velocity, 3D).

This reproduces, in matrix form, the position part of the model encoded
by `transfer_function_` in ekf.cpp, and also derives the extra terms
needed for the true Jacobian (`transfer_function_jacobian_`).
"""

import sympy as sp

sp.init_printing(use_unicode=True)

# ---------------------------------------------------------------------
# Symbols
# ---------------------------------------------------------------------
dt = sp.symbols('Delta_t', positive=True)              # time step

px, py, pz = sp.symbols('p_x p_y p_z')                  # position (world frame)
phi, theta, psi = sp.symbols('phi theta psi')            # roll, pitch, yaw
vx, vy, vz = sp.symbols('v_x v_y v_z')                   # linear velocity (body frame)
ax, ay, az = sp.symbols('a_x a_y a_z')                   # linear acceleration (body frame)

# ---------------------------------------------------------------------
# Elementary rotation matrices (body -> world), ZYX / yaw-pitch-roll
# convention, matching robot_localization's orientation convention.
# ---------------------------------------------------------------------
cphi, sphi = sp.cos(phi), sp.sin(phi)
ctheta, stheta = sp.cos(theta), sp.sin(theta)
cpsi, spsi = sp.cos(psi), sp.sin(psi)

Rz = sp.Matrix([[cpsi, -spsi, 0],
                [spsi,  cpsi, 0],
                [0,      0,   1]])

Ry = sp.Matrix([[ ctheta, 0, stheta],
                [ 0,      1, 0],
                [-stheta, 0, ctheta]])

Rx = sp.Matrix([[1, 0,     0],
                [0, cphi, -sphi],
                [0, sphi,  cphi]])

R = sp.simplify(Rz * Ry * Rx)   # body -> world

# print("=" * 70)
# print("Rotation matrix R(roll, pitch, yaw)  (body -> world)")
# print("=" * 70)
# sp.pprint(R)

# ---------------------------------------------------------------------
# Nonlinear position update:
#   p_k = p_{k-1} + R * ( v*dt + 1/2*a*dt^2 )
# ---------------------------------------------------------------------
p = sp.Matrix([px, py, pz])
v_b = sp.Matrix([vx, vy, vz])
a_b = sp.Matrix([ax, ay, az])

p_new = sp.simplify(p + R * (v_b * dt + sp.Rational(1, 2) * a_b * dt ** 2))

# print("\n" + "=" * 70)
# print("Nonlinear position update: p_k = p_(k-1) + R*(v*dt + 1/2*a*dt^2)")
# print("=" * 70)
# sp.pprint(sp.Eq(sp.Matrix([px, py, pz]), p_new))

# ---------------------------------------------------------------------
# transfer_function_ blocks: since p_new is LINEAR in v and a (for a
# fixed orientation), these Jacobians are exactly the coefficient
# blocks ekf.cpp writes into transfer_function_ at position rows /
# velocity & acceleration columns.
# ---------------------------------------------------------------------
F_pos_wrt_v = sp.simplify(p_new.jacobian(v_b))
F_pos_wrt_a = sp.simplify(p_new.jacobian(a_b))

# print("\n" + "=" * 70)
# print("transfer_function_ block  d(p_new)/d(v)   [position rows, velocity cols]")
# print("=" * 70)
# sp.pprint(F_pos_wrt_v)

# print("\n" + "=" * 70)
# print("transfer_function_ block  d(p_new)/d(a)   [position rows, accel cols]")
# print("=" * 70)
# sp.pprint(F_pos_wrt_a)

# ---------------------------------------------------------------------
# The TRUE Jacobian also needs derivatives w.r.t. orientation, since R
# itself depends on phi, theta, psi. transfer_function_ does NOT
# include these terms (it only multiplies state by the coefficient
# blocks above) -- this is exactly the extra piece that
# transfer_function_jacobian_ must add on top.
# ---------------------------------------------------------------------
J_pos_wrt_orientation = sp.simplify(p_new.jacobian(sp.Matrix([phi, theta, psi])))

# print("\n" + "=" * 70)
# print("Extra Jacobian terms  d(p_new)/d(phi,theta,psi)")
# print("(needed in transfer_function_jacobian_, absent from transfer_function_)")
# print("=" * 70)
# sp.pprint(J_pos_wrt_orientation)

# ---------------------------------------------------------------------
# Assemble the position rows of the full transfer_function_ matrix
# (columns ordered as: p(3), orientation(3), v(3), angvel(3), a(3))
# ---------------------------------------------------------------------
I3 = sp.eye(3)
Z3 = sp.zeros(3, 3)

F_position_rows = sp.Matrix.hstack(I3, Z3, F_pos_wrt_v, Z3, F_pos_wrt_a)

# print("\n" + "=" * 70)
# print("Position rows of transfer_function_ (15 columns:")
# print("[p | rpy | v | angvel | a])")
# print("=" * 70)
# sp.pprint(F_position_rows)

# ---------------------------------------------------------------------
# Isolate just the p_x equation (first row of p_new) and differentiate
# it with respect to vx. This single scalar derivative is exactly the
# entry transfer_function_(StateMemberX, StateMemberVx) in ekf.cpp.
# ---------------------------------------------------------------------
px_expr = p_new[0]                      # first row of p_new -> px
px_eq = sp.Eq(px, px_expr)

# print("\n" + "=" * 70)
# print("px equation only")
# print("=" * 70)
# sp.pprint(px_eq)

dpx_dvx = sp.simplify(sp.diff(px_expr, vx))

# print("\n" + "=" * 70)
# print("d(px)/d(vx)  -->  transfer_function_(StateMemberX, StateMemberVx)")
# print("=" * 70)
# sp.pprint(dpx_dvx)

# ---------------------------------------------------------------------
# Orientation update: body-frame angular velocities (wx=roll rate,
# wy=pitch rate, wz=yaw rate) map to Euler-angle rates through the
# standard Euler-rate transformation matrix E(phi, theta), then get
# integrated over dt. This matches the StateMemberRoll/Pitch/Yaw rows
# of transfer_function_ in ekf.cpp.
# ---------------------------------------------------------------------
wx, wy, wz = sp.symbols('omega_x omega_y omega_z')   # body angular velocity
ttheta = sp.tan(theta)

E = sp.Matrix([[1, sphi * ttheta, cphi * ttheta],
               [0, cphi,          -sphi],
               [0, sphi / ctheta, cphi / ctheta]])

orientation = sp.Matrix([phi, theta, psi])
omega_b = sp.Matrix([wx, wy, wz])

orientation_new = sp.simplify(orientation + E * omega_b * dt)

phi_eq   = sp.Eq(phi,   orientation_new[0])
theta_eq = sp.Eq(theta, orientation_new[1])
psi_eq   = sp.Eq(psi,   orientation_new[2])

phi_eq

# print("\n" + "=" * 70)
# print("Orientation equations only (roll, pitch, yaw)")
# print("=" * 70)
# sp.pprint(phi_eq)
# print()
# sp.pprint(theta_eq)
# print()
# sp.pprint(psi_eq)

In [8]:
theta_eq

In [9]:
psi_eq

In [10]:
orientation_new

⎡Δₜ⋅(ωₓ + ω_y⋅sin(φ)⋅tan(θ) + ω_z⋅cos(φ)⋅tan(θ)) + φ⎤
⎢                                                   ⎥
⎢         Δₜ⋅(ω_y⋅cos(φ) - ω_z⋅sin(φ)) + θ          ⎥
⎢                                                   ⎥
⎢      Δₜ⋅(ω_y⋅sin(φ) + ω_z⋅cos(φ)) + ψ⋅cos(θ)      ⎥
⎢      ───────────────────────────────────────      ⎥
⎣                      cos(θ)                       ⎦

In [11]:
# ---------------------------------------------------------------------
# Velocity update: body-frame velocity simply integrates body-frame
# acceleration over dt (constant-acceleration assumption). This
# matches the StateMemberVx/Vy/Vz rows of transfer_function_ in
# ekf.cpp, where the only nonzero off-diagonal entries are
# transfer_function_(StateMemberVx, StateMemberAx) = delta, etc.
# ---------------------------------------------------------------------
velocity = sp.Matrix([vx, vy, vz])
velocity_new = sp.simplify(velocity + a_b * dt)
 
vx_eq = sp.Eq(vx, velocity_new[0])
vy_eq = sp.Eq(vy, velocity_new[1])
vz_eq = sp.Eq(vz, velocity_new[2])
 
# print("\n" + "=" * 70)
# print("Velocity equations only (vx, vy, vz)")
# print("=" * 70)
# sp.pprint(vx_eq)
# print()
# sp.pprint(vy_eq)
# print()
# sp.pprint(vz_eq)

In [12]:
velocity_new

⎡ Δₜ⋅aₓ + vₓ ⎤
⎢            ⎥
⎢Δₜ⋅a_y + v_y⎥
⎢            ⎥
⎣Δₜ⋅a_z + v_z⎦

In [15]:
# =======================================================================
# Full nonlinear motion model f(x): assemble all 15 state equations.
# Angular velocity and linear acceleration are held constant between
# predictions (no formula changes them -- they persist unless acted on
# by a measurement update), matching ekf.cpp's identity rows for those
# state members.
# =======================================================================
omega_new = omega_b            # wx, wy, wz unchanged (constant angular velocity)
accel_new = a_b                # ax, ay, az unchanged (constant acceleration)
 
# Full state vector x, in robot_localization's order:
# [p(3), rpy(3), v(3), angvel(3), accel(3)]
x = sp.Matrix([px, py, pz,
               phi, theta, psi,
               vx, vy, vz,
               wx, wy, wz,
               ax, ay, az])
 
# Full nonlinear f(x): stack every updated state in the same order
f = sp.Matrix([p_new[0], p_new[1], p_new[2],
               orientation_new[0], orientation_new[1], orientation_new[2],
               velocity_new[0], velocity_new[1], velocity_new[2],
               omega_new[0], omega_new[1], omega_new[2],
               accel_new[0], accel_new[1], accel_new[2]])
 
# print("\n" + "=" * 70)
# print("Full nonlinear motion model  x_k = f(x_(k-1))")
# print("=" * 70)
# state_names = ['p_x', 'p_y', 'p_z', 'phi', 'theta', 'psi',
#                'v_x', 'v_y', 'v_z', 'omega_x', 'omega_y', 'omega_z',
#                'a_x', 'a_y', 'a_z']
# for name, expr in zip(state_names, f):
#     print(f"\n{name}_k =")
#     sp.pprint(expr)
 
# =======================================================================
# Full Jacobian F = df/dx  (this is transfer_function_jacobian_ in
# ekf.cpp -- the true EKF Jacobian used to propagate covariance).
# =======================================================================
F = f.jacobian(x)
 
# print("\n" + "=" * 70)
# print("Full Jacobian  F = df/dx  (15x15)")
# print("=" * 70)
# sp.pprint(F)



In [16]:
f

⎡Δₜ⋅(Δₜ⋅aₓ + 2⋅vₓ)⋅cos(ψ)⋅cos(θ)   Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅(sin(φ)⋅sin(θ)⋅cos(ψ) - ↪
⎢─────────────────────────────── + ─────────────────────────────────────────── ↪
⎢               2                                              2               ↪
⎢                                                                              ↪
⎢Δₜ⋅(Δₜ⋅aₓ + 2⋅vₓ)⋅sin(ψ)⋅cos(θ)   Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅(sin(φ)⋅sin(ψ)⋅sin(θ) + ↪
⎢─────────────────────────────── + ─────────────────────────────────────────── ↪
⎢               2                                              2               ↪
⎢                                                                              ↪
⎢                             Δₜ⋅(Δₜ⋅aₓ + 2⋅vₓ)⋅sin(θ)   Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅s ↪
⎢                           - ──────────────────────── + ───────────────────── ↪
⎢                                        2                               2     ↪
⎢                                                                              ↪
⎢                           

In [17]:
F

⎡                                                                              ↪
⎢         Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅(sin(φ)⋅sin(ψ) + sin(θ)⋅cos(φ)⋅cos(ψ))   Δₜ⋅(Δₜ⋅ ↪
⎢1  0  0  ────────────────────────────────────────────────────────── + ─────── ↪
⎢                                     2                                        ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢         Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅(-sin(φ)⋅cos(ψ) + sin(ψ)⋅sin(θ)⋅cos(φ))   Δₜ⋅(Δₜ ↪
⎢0  1  0  ─────────────────────────────────────────────────────────── - ────── ↪
⎢                                      2                                       ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                  Δₜ⋅(Δₜ⋅a_y + 2⋅v_y)⋅cos(φ)⋅cos(θ)   Δₜ⋅(Δₜ⋅ ↪
⎢0  0  1                    